<a href="https://colab.research.google.com/github/AMD2019/Python-for-Data-Science/blob/master/Education_desert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Install and Import Required Packages ---
!pip install geopandas rasterio shapely pyproj scikit-learn folium

import os
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import requests
import zipfile

# --- Setup: Paths and User-Editable Variables ---
DATA_DIR = '/content'      # Directory where you upload your data
OUTPUT_DIR = '/content/Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# User: update these with your actual files
# SCHOOL_SHP = os.path.join(DATA_DIR, 'schools_in_nepal_admin.shp') # Old school shapefile
SCHOOL_SHP = os.path.join(DATA_DIR, 'hotosm_npl_education_facilities_points_shp.shp') # New school shapefile with point geometries
ADMIN_SHP = os.path.join(DATA_DIR, 'nepal_admin_with_ids.shp')

print(f"Expecting school shapefile at: {SCHOOL_SHP}")
print(f"Expecting admin shapefile at: {ADMIN_SHP}")


# Distance threshold for education desert (in km)
DIST_THRESH = 3

# --- Download Population Raster for Nepal from WorldPop ---
# WorldPop: https://hub.worldpop.org/project/categories?id=3
POP_URL = "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/NPL/npl_ppp_2020.tif"
POP_TIF = os.path.join(DATA_DIR, 'npl_ppp_2020.tif')

# Download the file only if it doesn't exist
if not os.path.exists(POP_TIF):
    print(f"Downloading {POP_URL}...")
    try:
        response = requests.get(POP_URL, stream=True)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(POP_TIF, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Download complete: {POP_TIF}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
else:
    print(f"File already exists: {POP_TIF}")


# --- Read Vector Data ---
try:
    schools_gdf = gpd.read_file(SCHOOL_SHP)
    admin_gdf = gpd.read_file(ADMIN_SHP)
except Exception as e:
    print(f"Error reading shapefiles: {e}")
    print("Please ensure you have uploaded 'hotosm_npl_education_facilities_points_shp.shp' and 'nepal_admin_with_ids.shp' to the '/content' directory.")
    # Exit or raise the error if files are critical to proceed
    raise


# --- Filter admin data to Karanali province ---
# Assuming 'ADM1_EN' is the correct column name for province names
try:
    karanali_province = admin_gdf[admin_gdf['ADM1_EN'] == 'Karnali'].copy()
    if karanali_province.empty:
        print("Warning: 'Karnali' province not found in 'ADM1_EN' column. Please check the spelling and casing.")
except KeyError:
    print("Error: 'ADM1_EN' column not found in admin shapefile. Please check the column names.")
    raise # Re-raise the error if the column is critical

# --- Filter schools to include only specified amenity types ---
amenity_types = ['college', 'kindergarten', 'school', 'university']
# Note: The column name for amenity types might be different in the new shapefile.
# Assuming it's still 'amenity' for now, but this might need adjustment.
if 'amenity' in schools_gdf.columns:
    schools_gdf = schools_gdf[schools_gdf['amenity'].isin(amenity_types)].copy()
else:
    print("Warning: 'amenity' column not found in the new school shapefile. Skipping amenity type filtering.")


# --- Filter schools to include only Point geometries ---
# This should now work as the new shapefile is expected to have points
schools_gdf = schools_gdf[schools_gdf.geometry.type == 'Point'].copy()


# --- Read Population Raster ---
try:
    pop_src = rasterio.open(POP_TIF)
except Exception as e:
    print(f"Error opening population raster file: {e}")
    raise

# Determine the target CRS for distance calculation (UTM for Nepal)
utm_crs = "EPSG:32645" # UTM zone 45N for Nepal

# Ensure schools_gdf is in the target UTM CRS for distance calculations
if schools_gdf.crs != utm_crs:
    print(f"Reprojecting schools_gdf from {schools_gdf.crs} to {utm_crs}")
    schools_gdf_utm = schools_gdf.to_crs(utm_crs)
else:
    schools_gdf_utm = schools_gdf

# Extract school coordinates in UTM
# Check if schools_gdf_utm is empty before creating the BallTree
if schools_gdf_utm.empty:
    print("Error: No school points found after filtering and reprojection. Cannot build BallTree.")
    all_pop_points_gdf = [] # Ensure this list is empty to skip subsequent steps
else:
    school_coords_utm = np.array(list(zip(schools_gdf_utm.geometry.x, schools_gdf_utm.geometry.y)))
    tree = BallTree(school_coords_utm, metric='euclidean')

    all_pop_points_gdf = []

    # --- Iterate and Process by Administrative Unit (within Karanali province) ---
    if not karanali_province.empty:
        # Ensure the karanali_province GeoDataFrame is in the raster's CRS before iterating
        if karanali_province.crs != pop_src.crs:
            print(f"Reprojecting karanali_province from {karanali_province.crs} to {pop_src.crs}")
            karanali_province = karanali_province.to_crs(pop_src.crs)

        for index, row in karanali_province.iterrows():
            try:
                # Get the geometry of the current administrative unit
                geom = row.geometry

                # Mask the raster with the current geometry
                out_image, out_transform = mask(pop_src, [geom], crop=True, nodata=pop_src.nodata)

                pop_img_chunk = out_image[0] # Assuming single band
                pop_transform_chunk = out_transform

                # Extract Population Grid Points with Population > 0
                rows, cols = np.where(pop_img_chunk > 0)

                # If no population points in this chunk, skip
                if rows.size == 0:
                    admin_name = row.get('ADM3_EN') or row.get('ADM2_EN', f"Index: {index}")
                    print(f"No populated points found in admin unit: {admin_name}. Skipping distance calculation.")
                    continue

                # Convert to spatial coordinates (x, y)
                xs, ys = rasterio.transform.xy(pop_transform_chunk, rows, cols)

                # Create a pandas DataFrame with 'x', 'y', and 'pop' columns
                pop_points_chunk = pd.DataFrame({'x': xs, 'y': ys, 'pop': pop_img_chunk[rows, cols]})

                # Convert to a GeoDataFrame
                # Get the CRS from the original raster source for this chunk
                pop_gdf_chunk = gpd.GeoDataFrame(pop_points_chunk, geometry=gpd.points_from_xy(xs, ys), crs=pop_src.crs)

                # Project the current population points chunk to the target UTM CRS for distance calculation
                pop_gdf_chunk_utm = pop_gdf_chunk.to_crs(utm_crs)
                pop_coords_chunk_utm = np.array(list(zip(pop_gdf_chunk_utm.geometry.x, pop_gdf_chunk_utm.geometry.y)))

                # Query the BallTree to find the distance to the nearest school
                distances, indices = tree.query(pop_coords_chunk_utm, k=1)

                # Add the calculated distances (in meters) as a new column (convert to km)
                pop_gdf_chunk['dist_to_school_km'] = (distances.flatten()) / 1000

                # Add the administrative unit's identifier
                for col in ['ADM0_EN', 'ADM0_PCODE', 'ADM1_EN', 'ADM1_PCODE', 'ADM2_EN', 'ADM2_PCODE', 'ADM3_EN', 'ADM3_PCODE']:
                     if col in row:
                          pop_gdf_chunk[col] = row[col]
                     else:
                          pop_gdf_chunk[col] = None # Add column with None if not present in this level


                # Append the population points GeoDataFrame chunk to the list
                all_pop_points_gdf.append(pop_gdf_chunk)

                admin_name = row.get('ADM3_EN') or row.get('ADM2_EN', f"Index: {index}")
                print(f"Processed population points and distances for admin unit: {admin_name}")


            except Exception as e:
                admin_name = row.get('ADM3_EN') or row.get('ADM2_EN', f"Index: {index}")
                print(f"Error processing admin unit {admin_name}: {e}")
                continue # Continue to the next administrative unit even if one fails

    # Close the population raster file
    pop_src.close()


# --- Combine Results ---
if all_pop_points_gdf:
    final_pop_gdf = pd.concat(all_pop_points_gdf, ignore_index=True)
    print(f"Concatenated data from {len(all_pop_points_gdf)} chunks into a single GeoDataFrame with {len(final_pop_gdf)} points.")

    # --- Identify Education Deserts ---
    final_pop_gdf['education_desert'] = final_pop_gdf['dist_to_school_km'] > DIST_THRESH

    # --- Simple Visualization ---
    fig, ax = plt.subplots(figsize=(15,10)) # Increased figsize
    # Plot the boundary of the processed province
    karanali_province.boundary.plot(ax=ax, color='black')
    # Plot education deserts
    final_pop_gdf[final_pop_gdf['education_desert']].plot(ax=ax, color='yellow', markersize=0.5, label='Education Desert')
    # Plot schools (reproject schools_gdf to the plot's CRS if necessary, assuming plot is in karanali_province CRS)
    # Filter schools_gdf for points within the extent of Karanali province for plotting efficiency
    schools_gdf_karnali_extent = schools_gdf.cx[karanali_province.total_bounds[0]:karanali_province.total_bounds[2], karanali_province.total_bounds[1]:karanali_province.total_bounds[3]]

    if schools_gdf_karnali_extent.crs != karanali_province.crs:
        schools_gdf_karnali_extent.to_crs(karanali_province.crs).plot(ax=ax, color='blue', markersize=2, label='Schools')
    else:
        schools_gdf_karnali_extent.plot(ax=ax, color='blue', markersize=2, label='Schools')


    plt.legend()
    plt.title('Education Deserts in Karanali Province (>{}km from nearest school)'.format(DIST_THRESH))
    ax.set_axis_off() # Turn off the axes
    ax.set_aspect('equal', adjustable='box') # Set equal aspect ratio
    plt.show()

    # --- Save Output as GeoPackage ---
    # Ensure the final_pop_gdf is in a suitable CRS for saving (e.g., WGS84)
    if final_pop_gdf.crs != "EPSG:4326":
        final_pop_gdf = final_pop_gdf.to_crs("EPSG:4326")

    final_pop_gdf.to_file(os.path.join(OUTPUT_DIR, 'karanali_education_deserts.gpkg'), driver='GPKG')
    print(f"Saved education desert data for Karanali province to {os.path.join(OUTPUT_DIR, 'karanali_education_deserts.gpkg')}")

else:
    print("No population points were processed for Karanali province.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 77.7 MB/s eta 0:00:00
Expecting school shapefile at: /content/hotosm_npl_education_facilities_points_shp.shp
Expecting admin shapefile at: /content/nepal_admin_with_ids.shp
Download complete: /content/npl_ppp_2020.tif
Error reading shapefiles: /content/hotosm_npl_education_facilities_points_shp.shp: No such file or directory
Please ensure you have uploaded 'hotosm_npl_education_facilities_points_shp.shp' and 'nepal_admin_with_ids.shp' to the '/content' directory.


DataSourceError: /content/hotosm_npl_education_facilities_points_shp.shp: No such file or directory

In [ ]:
# prompt: From above analysis can you generate 3D map of the population points of Karnali province in the education deserts

# --- 3D Visualization ---
!pip install plotly # Install plotly if not already installed

import plotly.graph_objects as go

# Filter for education deserts in Karanali province
education_deserts_gdf = final_pop_gdf[final_pop_gdf['education_desert']].copy()

if not education_deserts_gdf.empty:
    # Extract coordinates and population for 3D plot
    x = education_deserts_gdf.geometry.x
    y = education_deserts_gdf.geometry.y
    z = education_deserts_gdf['pop']

    # Create the 3D scatter plot
    fig = go.Figure(data=[go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode='markers',
        marker=dict(
            size=2, # Adjust marker size as needed
            color=z, # Color by population
            colorscale='Viridis', # Choose a colorscale
            opacity=0.8
        )
    )])

    # Update layout
    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=0),
        title='3D Population Distribution in Education Deserts (Karnali Province)',
        scene = dict(
            xaxis_title='Longitude',
            yaxis_title='Latitude',
            zaxis_title='Population'
        )
    )

    # Show the plot
    fig.show()

else:
    print("No education desert points found in Karanali province to generate 3D plot.")



Buffered data was truncated after reaching the output size limit.